In [1]:

import pandas as pd
import numpy as np
import os

# ================= 1. 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_PATH = "primekg_ad_only.csv"  # 保持文件名不变，方便后续脚本直接用

# [新增] 黑名单：基于侦察结果，这些节点连接数过高且无特异性，必须剔除！
BLACKLIST_NODES = [
    "multi-cellular organism", 
    "small intestine", "intestine", "colon", "stomach", "esophagus", # 消化道通用部位
    "testis", "fallopian tube", "prostate gland", "myometrium", "female gonad", # 生殖系统通用部位
    "spleen", "adrenal gland", "thyroid gland", "saliva-secreting gland", # 通用腺体
    "adult mammalian kidney", "cortex of kidney", # 肾脏通用
    "dorsolateral prefrontal cortex", "prefrontal cortex", "frontal cortex" # 虽然是大脑，但过于宽泛，容易变成Hub
]

# [扩充] 关键词军团：包含 AD + 经验证的代谢核心词
KEYWORDS = [
    # --- 核心 AD & 认知 ---
    "Alzheimer", "Dementia", "Memory", "Cognitive", 
    "Amyloid", "Tau", "Neurofibrillary",
    
    # --- 核心 代谢 & 血管 (经验证有效) ---
    "Insulin",      # 胰岛素 (关联 Insulin resistance)
    "Glucose",      # 血糖 (关联 Abnormal glucose homeostasis)
    "Lipid",        # 脂质 (关联 Lipid accumulation)
    "Cholesterol",  # 胆固醇
    "Hypertension", # 高血压 (关联 Pulmonary arterial hypertension)
    "Diabetes"      # 糖尿病
]

# 允许的类型 (严格剔除 drug 和 exposure)
VALID_TYPES = [
    'disease',
    'gene/protein',
    'effect/phenotype',
    'pathway',
    'biological_process',
    'anatomy'
]

def main():
    print(f"🔹 1. 正在加载原始 PrimeKG ({RAW_KG_PATH}) ...")
    # 读取所有数据
    df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    print(f"   原始数据量: {len(df)}")

    # ================= 步骤 A: 预处理 (黑名单 & 类型) =================
    print(f"🔹 2. 执行清洗任务...")
    
    # 1. 类型清洗：只保留 VALID_TYPES (剔除 Drug, Exposure, Cellular component等)
    mask_type = (df['x_type'].isin(VALID_TYPES)) & (df['y_type'].isin(VALID_TYPES))
    
    # 2. 关系清洗：剔除药物相关关系 (双重保险)
    mask_rel = ~df['relation'].isin(['indication', 'contraindication', 'drug_drug', 'off-label use'])
    
    # 3. 黑名单清洗：剔除那些连接数几万的通用解剖节点
    # 注意：这里我们排除掉 x_name 或 y_name 在黑名单里的行
    mask_blacklist = (~df['x_name'].isin(BLACKLIST_NODES)) & (~df['y_name'].isin(BLACKLIST_NODES))
    
    # 应用初步清洗
    df_clean = df[mask_type & mask_rel & mask_blacklist].copy()
    print(f"   🧹 清洗后剩余基础数据: {len(df_clean)} 条 (已去除药物、Exposure及黑名单Hub节点)")

    # ================= 步骤 B: 关键词筛选 =================
    print(f"🔹 3. 正在根据扩充后的关键词筛选: {KEYWORDS} ...")

    # 正则表达式 "Alzheimer|Insulin|..."
    pattern = '|'.join(KEYWORDS)

    # 只要两端任意一端包含关键词，就保留
    mask_keyword = (df_clean['x_name'].str.contains(pattern, case=False, na=False)) | \
                   (df_clean['y_name'].str.contains(pattern, case=False, na=False))

    subset = df_clean[mask_keyword].copy()

    if len(subset) == 0:
        print("❌ 错误：没有找到任何符合条件的数据！")
        return

    print(f"   ✅ 筛选出核心知识: {len(subset)} 条")
    
    # ================= 步骤 C: 补充 PPI (基因互作) =================
    print("\n🔹 4. 正在补充基因内部互作 (PPI) 以连接 AD 与 代谢网络...")

    # 1. 拿到刚才筛选出的所有基因 (比如 APP, APOE, INS, AKT1...)
    related_genes = set(subset[subset['y_type'] == 'gene/protein']['y_name']) | \
                    set(subset[subset['x_type'] == 'gene/protein']['x_name'])

    print(f"   🧬 识别到 {len(related_genes)} 个关键基因，正在查找互作网络...")

    if len(related_genes) > 0:
        # 2. 在清洗过的大表(df_clean)里找：两头都是这些基因的 protein_protein 关系
        # 注意：必须用 df_clean，确保不会把药物-蛋白关系加进来
        mask_ppi = (
                (df_clean['x_name'].isin(related_genes)) &
                (df_clean['y_name'].isin(related_genes)) &
                (df_clean['relation'] == 'protein_protein')
        )
        ppi_df = df_clean[mask_ppi]
        print(f"   🕸 成功补充了 {len(ppi_df)} 条 PPI 边")

        # 3. 合并
        final_df = pd.concat([subset, ppi_df]).drop_duplicates()
    else:
        final_df = subset
        print("   => 未发现相关基因，跳过 PPI 补充。")

    # ================= 步骤 D: 保存 =================
    print(f"\n🔹 5. 保存最终 AD-Metabolic 图谱至: {OUTPUT_PATH}")
    final_df.to_csv(OUTPUT_PATH, index=False)

    print("-" * 50)
    print(f"🎉 知识图谱构建完成！总条目: {len(final_df)}")
    print(f"   包含内容: AD病理 + 认知症状 + {', '.join(KEYWORDS[7:])} 等代谢机制")
    print(f"   已排除: 所有药物节点, {len(BLACKLIST_NODES)} 个高频通用干扰节点")
    print("-" * 50)
    
    # 预览几条新加入的代谢知识，让你确认
    print("\n👀 [新知识预览] 看看有没有混入奇怪的东西？")
    # 筛选包含 "Insulin" 或 "Lipid" 的行展示一下
    preview_mask = final_df['x_name'].str.contains("Insulin|Lipid", case=False) | \
                   final_df['y_name'].str.contains("Insulin|Lipid", case=False)
    print(final_df[preview_mask][['x_name', 'relation', 'y_name']].head(5))

if __name__ == "__main__":
    main()

🔹 1. 正在加载原始 PrimeKG (kg.csv) ...
   原始数据量: 8100498
🔹 2. 执行清洗任务...
   🧹 清洗后剩余基础数据: 4148958 条 (已去除药物、Exposure及黑名单Hub节点)
🔹 3. 正在根据扩充后的关键词筛选: ['Alzheimer', 'Dementia', 'Memory', 'Cognitive', 'Amyloid', 'Tau', 'Neurofibrillary', 'Insulin', 'Glucose', 'Lipid', 'Cholesterol', 'Hypertension', 'Diabetes'] ...
   ✅ 筛选出核心知识: 26078 条

🔹 4. 正在补充基因内部互作 (PPI) 以连接 AD 与 代谢网络...
   🧬 识别到 2771 个关键基因，正在查找互作网络...
   🕸 成功补充了 41870 条 PPI 边

🔹 5. 保存最终 AD-Metabolic 图谱至: primekg_ad_only.csv
--------------------------------------------------
🎉 知识图谱构建完成！总条目: 67650
   包含内容: AD病理 + 认知症状 + Insulin, Glucose, Lipid, Cholesterol, Hypertension, Diabetes 等代谢机制
   已排除: 所有药物节点, 20 个高频通用干扰节点
--------------------------------------------------

👀 [新知识预览] 看看有没有混入奇怪的东西？
        x_name           relation              y_name
3062679  ACACB  phenotype_protein  Insulin resistance
3062680  ADRB2  phenotype_protein  Insulin resistance
3062681    AHR  phenotype_protein  Insulin resistance
3062682     AR  phenotype_protein  Insulin res

In [2]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from collections import Counter

# ================= 1. 配置区 =================
# AIBL 本地数据文件 (文件名大小写需与实际文件完全一致)
CSV_FILES = ['AD.csv', 'MCI.csv', 'NC.csv']

# 指向第一步生成的通用生物底座 (确保文件名与上一步生成的一致)
PRIMEKG_PATH = "primekg_ad_only.csv"

# 输出文件
OUTPUT_TRIPLETS = "aibl_knowledge_triplets.csv"
OUTPUT_E2ID = "aibl_kg_entity2id.json"
OUTPUT_R2ID = "aibl_kg_relation2id.json"

# ================= 2. 映射逻辑定义 =================

base_col_map = {
    "age": "Concept:Age",
    "gender": "Concept:Sex"
}

def load_primekg():
    """读取筛选后的 PrimeKG 生物学子图"""
    print(f"🔹 1. 正在读取 PrimeKG 生物底座: {PRIMEKG_PATH} ...")
    if not os.path.exists(PRIMEKG_PATH):
        print(f"❌ 错误：找不到 {PRIMEKG_PATH}，请先运行 filter_primekg.py")
        return []

    triplets = []
    try:
        df = pd.read_csv(PRIMEKG_PATH, low_memory=False)
        
        # 自动识别列名 (兼容不同版本 CSV)
        cols = df.columns
        x_col = next((c for c in cols if 'x_name' in c or 'head' in c), 'x_name')
        y_col = next((c for c in cols if 'y_name' in c or 'tail' in c), 'y_name')
        r_col = next((c for c in cols if 'relation' in c), 'relation')

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing PrimeKG"):
            # 统一加上 PrimeKG 前缀，防止名字冲突
            h = "PrimeKG:" + str(row[x_col])
            t = "PrimeKG:" + str(row[y_col])
            r = str(row[r_col])
            triplets.append((h, r, t))

        print(f"   ✅ 已加载生物医学知识: {len(triplets)} 条 (包含 AD 及 代谢网络)")
    except Exception as e:
        print(f"   ❌ 读取 PrimeKG 失败: {e}")
    return triplets


def process_aibl_files():
    """处理 AIBL CSV 文件"""
    print(f"🔹 2. 正在处理 AIBL 临床数据...")
    local_triplets = []
    patient_ids = set()
    
    # 统计计数器
    stats = {
        "apoe_carriers": 0,
        "mri_info": 0,
        "mmse_scores": 0
    }

    for file_name in CSV_FILES:
        if not os.path.exists(file_name):
            print(f"   ⚠️ 跳过缺失文件: {file_name}")
            continue

        print(f"      -> 正在解析: {file_name} ...")
        df = pd.read_csv(file_name)

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"   Reading {file_name}"):
            # 1. 构建病人 ID
            if 'filename' in row and pd.notna(row['filename']):
                s_id = str(row['filename']).strip()
            else:
                continue

            patient_id = f"Patient:{s_id}"
            patient_ids.add(patient_id)

            # 2. 基础属性
            for col, relation_name in base_col_map.items():
                if col in row and pd.notna(row[col]):
                    val = str(row[col])
                    target_node = f"{relation_name}:{val}"
                    local_triplets.append((patient_id, "has_attribute", target_node))

            # 3. 影像参数 (Tesla)
            if 'Tesla' in row and pd.notna(row['Tesla']):
                try:
                    tesla_val = float(row['Tesla'])
                    tesla_node = f"Concept:MRI_Strength:{tesla_val}T"
                    local_triplets.append((patient_id, "scanned_with", tesla_node))
                    stats["mri_info"] += 1
                except: pass

            # 4. 核心基因 (APOE) -> 这是连接 PrimeKG 的关键桥梁
            if 'apoe' in row and pd.notna(row['apoe']):
                try:
                    apoe_val = int(float(row['apoe']))
                    if apoe_val > 0: # 携带者
                        gene_node = "Gene:APOE_e4_Carrier"
                        local_triplets.append((patient_id, "has_gene_risk", gene_node))
                        # ★ 关键：手动建立与 PrimeKG 的连接 ★
                        local_triplets.append((gene_node, "risk_factor_for", "PrimeKG:Alzheimer disease"))
                        local_triplets.append((gene_node, "is_variant_of", "PrimeKG:APOE")) 
                        stats["apoe_carriers"] += 1
                except: pass

            # 5. 认知评分
            if 'mmse' in row and pd.notna(row['mmse']):
                try:
                    mmse = int(float(row['mmse']))
                    mmse_node = f"Test:MMSE:{mmse}"
                    local_triplets.append((patient_id, "has_mmse_score", mmse_node))
                    stats["mmse_scores"] += 1
                except: pass
            
            # CDR & Logical Memory
            if 'cdr' in row and pd.notna(row['cdr']):
                local_triplets.append((patient_id, "has_cdr_score", f"Test:CDR:{float(row['cdr'])}"))
            
            if 'lm_imm' in row and pd.notna(row['lm_imm']):
                local_triplets.append((patient_id, "has_memory_score", f"Test:LM_Immediate:{int(float(row['lm_imm']))}"))
                
            if 'lm_del' in row and pd.notna(row['lm_del']):
                local_triplets.append((patient_id, "has_memory_score", f"Test:LM_Delayed:{int(float(row['lm_del']))}"))

    print(f"   ✅ AIBL 处理完成: 生成 {len(local_triplets)} 条临床关系")
    print(f"      (统计: {stats['mri_info']} 份MRI参数, {stats['apoe_carriers']} 位APOE携带者, {stats['mmse_scores']} 份MMSE评分)")
    return local_triplets

def analyze_graph_statistics(df):
    """打印详细的图谱体检报告"""
    print("\n" + "="*60)
    print("📊 [最终异构图谱深度体检报告]")
    print("="*60)
    
    # 1. 节点类型分析 (根据前缀)
    all_nodes = list(df['head']) + list(df['tail'])
    unique_nodes = set(all_nodes)
    
    # 简单的分类器
    node_types = []
    for n in unique_nodes:
        if str(n).startswith("Patient:"): node_types.append("Patient (病人)")
        elif str(n).startswith("PrimeKG:"): node_types.append("PrimeKG (生物知识)")
        elif str(n).startswith("Gene:"): node_types.append("Gene (特定基因)")
        elif str(n).startswith("Test:"): node_types.append("Test (认知量表)")
        elif str(n).startswith("Concept:"): node_types.append("Concept (人口学/参数)")
        else: node_types.append("Other (其他)")
        
    type_counts = Counter(node_types)
    print(f"1. 节点构成 (总数: {len(unique_nodes)}):")
    for ntype, count in type_counts.most_common():
        print(f"   - {ntype:<20} : {count}")
        
    # 2. 关系分布
    print(f"\n2. 关系分布 (Top 15):")
    print(df['relation'].value_counts().head(15))S
    
    # 3. 桥梁检查
    print(f"\n3. 关键连接检查:")
    # 检查是否有病人连到了 APOE 基因
    bridge_mask = (df['head'].str.contains("Patient")) & (df['tail'].str.contains("APOE"))
    bridge_count = len(df[bridge_mask])
    print(f"   - 病人 -> APOE 基因连接数: {bridge_count} 条")
    
    # 检查是否有基因连到了代谢 (Insulin/Lipid)
    # 注意：这里需要在 PrimeKG 节点里找
    metabolic_mask = df['tail'].str.contains("Insulin|Lipid|Glucose", case=False)
    metabolic_count = len(df[metabolic_mask])
    print(f"   - 指向代谢相关节点(Insulin/Lipid等)的连接数: {metabolic_count} 条")

    print("="*60 + "\n")

def main():
    # 1. 加载
    kg_triplets = load_primekg()
    aibl_triplets = process_aibl_files()

    if not aibl_triplets:
        print("❌ 严重警告：未生成任何 AIBL 临床数据！")
        return

    # 2. 合并
    all_triplets = kg_triplets + aibl_triplets

    # 3. 保存与去重
    df_out = pd.DataFrame(all_triplets, columns=['head', 'relation', 'tail'])

    before_dedup = len(df_out)
    df_out.drop_duplicates(inplace=True)
    after_dedup = len(df_out)

    print(f"✂️  执行去重操作: {before_dedup} -> {after_dedup} (移除 {before_dedup - after_dedup})")

    # 4. 执行深度体检 (这是新加的)
    analyze_graph_statistics(df_out)

    # 5. 保存文件
    df_out.to_csv(OUTPUT_TRIPLETS, index=False)
    print(f"💾 最终 AIBL 异构图谱已保存至: {OUTPUT_TRIPLETS}")

    # 6. 生成 ID 映射
    print("🔹 正在生成 Entity/Relation ID 映射表...")
    entities = sorted(list(set(df_out['head']) | set(df_out['tail'])))
    relations = sorted(list(set(df_out['relation'])))

    ent2id = {e: i for i, e in enumerate(entities)}
    rel2id = {r: i for i, r in enumerate(relations)}

    with open(OUTPUT_E2ID, 'w') as f: json.dump(ent2id, f)
    with open(OUTPUT_R2ID, 'w') as f: json.dump(rel2id, f)

    print(f"   映射表已保存: {OUTPUT_E2ID}, {OUTPUT_R2ID}")

    # 简单验证
    print("\n🔍 验证环节: 检查是否有 NC 病人")
    nc_count = sum(1 for e in entities if "Patient:" in e and "S236160" in e)
    print(f"   包含示例 NC 病人 (S236160)? {'✅ 是' if nc_count > 0 else '❌ 否'}")

if __name__ == "__main__":
    main()

🔹 1. 正在读取 PrimeKG 生物底座: primekg_ad_only.csv ...


Parsing PrimeKG: 100%|████████████████████████████████████████████████████████| 67650/67650 [00:05<00:00, 12461.48it/s]


   ✅ 已加载生物医学知识: 67650 条 (包含 AD 及 代谢网络)
🔹 2. 正在处理 AIBL 临床数据...
      -> 正在解析: AD.csv ...


   Reading AD.csv: 100%|█████████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 5085.12it/s]


      -> 正在解析: MCI.csv ...


   Reading MCI.csv: 100%|████████████████████████████████████████████████████████████| 86/86 [00:00<00:00, 4692.59it/s]


      -> 正在解析: NC.csv ...


   Reading NC.csv: 100%|███████████████████████████████████████████████████████████| 358/358 [00:00<00:00, 4685.67it/s]


   ✅ AIBL 处理完成: 生成 3714 条临床关系
      (统计: 516 份MRI参数, 34 位APOE携带者, 516 份MMSE评分)
✂️  执行去重操作: 71364 -> 71298 (移除 66)

📊 [最终异构图谱深度体检报告]
1. 节点构成 (总数: 7870):
   - PrimeKG (生物知识)       : 7239
   - Patient (病人)         : 516
   - Test (认知量表)          : 73
   - Concept (人口学/参数)     : 41
   - Gene (特定基因)          : 1

2. 关系分布 (Top 15):
relation
protein_protein               41870
disease_phenotype_positive     8024
bioprocess_protein             7322
bioprocess_bioprocess          2930
disease_protein                2698
pathway_protein                1648
disease_disease                1454
has_attribute                  1032
has_memory_score               1032
phenotype_phenotype             648
scanned_with                    516
has_mmse_score                  516
has_cdr_score                   516
anatomy_protein_present         472
phenotype_protein               306
Name: count, dtype: int64

3. 关键连接检查:
   - 病人 -> APOE 基因连接数: 34 条
   - 指向代谢相关节点(Insulin/Lipid等)的连接数: 6522 条

💾 最终 AIBL 异构图谱

In [1]:
#知识图谱剪枝与优化
import pandas as pd
import networkx as nx
import os
import shutil

# ================= ⚡ 配置区 =================
# 目标文件 (保持不变，直接覆盖清洗)
TARGET_FILE = 'aibl_knowledge_triplets.csv'

# 保护名单：这些前缀的节点绝对不能删，哪怕它们只有一条连线
PROTECTED_PREFIXES = [
    "Patient:",   # 病人
    "Gene:",      # 基因 (桥梁)
    "Concept:",   # 年龄/性别
    "Test:",      # MMSE/CDR
    "Symptom:"    # 症状
]

def analyze_graph(df, stage="清洗前"):
    """📊 打印图谱健康状况"""
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # 计算连通子图数量 (理想情况下应该很少，说明知识都连在一起)
    components = list(nx.connected_components(G))
    num_components = len(components)
    largest_comp = len(max(components, key=len)) if components else 0
    
    print(f"--- 🏥 [{stage}] 体检报告 ---")
    print(f"   节点: {num_nodes} | 边: {num_edges}")
    print(f"   连通孤岛数: {num_components} (越少越好)")
    print(f"   最大连通子图节点数: {largest_comp}")
    return G

def main():
    if not os.path.exists(TARGET_FILE):
        print(f"❌ 找不到文件: {TARGET_FILE}")
        return

    print(f"🚀 正在加载 {TARGET_FILE} ...")
    df = pd.read_csv(TARGET_FILE)
    
    # 1. 术前体检
    analyze_graph(df, "清洗前")
    
    # 2. 创建备份 (安全第一)
    backup_file = TARGET_FILE + ".bak"
    shutil.copy(TARGET_FILE, backup_file)
    print(f"📦 已创建备份文件: {backup_file} (如果出问题可以用这个恢复)")

    # ================= 核心清洗逻辑 =================
    print("\n🧹 开始执行原地净化...")
    
    # 构建图结构
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    nodes_to_keep = set()
    
    # --- 策略 A: 移除无法到达病人的“死知识” ---
    print("   1. 正在切除与病人断连的孤立知识岛屿...")
    components = nx.connected_components(G)
    
    for comp in components:
        # 检查这个岛屿里有没有病人
        has_patient = any(str(n).startswith("Patient:") for n in comp)
        # 检查有没有关键基因 (Gene:APOE等)，防止把还没连上病人的关键基因误删
        has_gene = any(str(n).startswith("Gene:") for n in comp)
        
        if has_patient or has_gene:
            nodes_to_keep.update(comp)
        else:
            # 这个岛屿里全是 PrimeKG 的边缘知识，跟我们没关系，丢弃
            pass
            
    # --- 策略 B: 修剪末梢悬挂节点 (Dead Ends) ---
    print("   2. 正在修剪低效的末梢节点...")
    # 只在保留下来的节点里修剪
    G_sub = G.subgraph(list(nodes_to_keep)).copy()
    
    nodes_to_remove = []
    for node in G_sub.nodes():
        # 如果是受保护节点，跳过
        if any(str(node).startswith(p) for p in PROTECTED_PREFIXES):
            continue
            
        # 如果度为1 (悬挂)，且不是受保护节点 -> 视为噪声
        if G_sub.degree(node) == 1:
            nodes_to_remove.append(node)
            
    final_nodes = set(G_sub.nodes()) - set(nodes_to_remove)
    
    # ================= 保存结果 =================
    print("   3. 正在重组数据...")
    
    # 筛选只包含 final_nodes 的三元组
    mask = df['head'].isin(final_nodes) & df['tail'].isin(final_nodes)
    df_clean = df[mask].copy()
    
    # 3. 术后体检
    analyze_graph(df_clean, "清洗后")
    
    # 覆盖原文件
    df_clean.to_csv(TARGET_FILE, index=False)
    print(f"\n✅ 成功！原文件 {TARGET_FILE} 已更新。")
    print(f"   (原数据量 {len(df)} -> 现数据量 {len(df_clean)})")

if __name__ == "__main__":
    main()

🚀 正在加载 aibl_knowledge_triplets.csv ...
--- 🏥 [清洗前] 体检报告 ---
   节点: 7870 | 边: 37467
   连通孤岛数: 51 (越少越好)
   最大连通子图节点数: 7615
📦 已创建备份文件: aibl_knowledge_triplets.csv.bak (如果出问题可以用这个恢复)

🧹 开始执行原地净化...
   1. 正在切除与病人断连的孤立知识岛屿...
   2. 正在修剪低效的末梢节点...
   3. 正在重组数据...
--- 🏥 [清洗后] 体检报告 ---
   节点: 5040 | 边: 34647
   连通孤岛数: 1 (越少越好)
   最大连通子图节点数: 5040

✅ 成功！原文件 aibl_knowledge_triplets.csv 已更新。
   (原数据量 71298 -> 现数据量 65658)
👉 你现在可以直接运行 train_aibl_distmult.py，无需修改任何代码。


In [2]:
#train_aibl_distmult
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
import json
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

# ================= ⚡ 配置区 (已修改为 AIBL) =================
# 1. 输入文件 (对应 build_aibl_primekg.py 的输出)
TRIPLETS_FILE = 'aibl_knowledge_triplets.csv'    # <--- 修改点 1
ENTITY2ID_FILE = 'aibl_kg_entity2id.json'        # <--- 修改点 2
RELATION2ID_FILE = 'aibl_kg_relation2id.json'    # <--- 修改点 3

# 2. 输出文件 (训练好的 Embeddings)
OUTPUT_EMBED = 'aibl_kg_embeddings.npy'          # <--- 修改点 4

# 3. 训练超参数 (保持与 ADNI 一致，方便横向对比)
EMBED_DIM = 128
NUM_EPOCHS = 100
BATCH_SIZE = 512
LR = 0.005
TRAIN_RATIO = 0.9
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 [AIBL] 训练设备: {DEVICE}")


# ================= 🛠️ 数据加载类 (通用) =================
class KGDataset(Dataset):
    def __init__(self, triplets_file, entity2id, relation2id):
        print(f"    正在读取图谱文件: {triplets_file} ...")

        try:
            df = pd.read_csv(triplets_file)
        except Exception as e:
            print(f"❌ 读取 CSV 失败: {e}")
            self.triplets = []
            return

        print(f"    ✅ 成功加载原始数据: {len(df)} 行")

        self.triplets = []
        skipped = 0

        # 将字符串转换为 ID
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Indexing Data"):
            try:
                h_token = str(row['head']).strip()
                r_token = str(row['relation']).strip()
                t_token = str(row['tail']).strip()

                h = entity2id.get(h_token)
                r = relation2id.get(r_token)
                t = entity2id.get(t_token)

                if h is not None and r is not None and t is not None:
                    self.triplets.append((h, r, t))
                else:
                    skipped += 1
            except Exception:
                skipped += 1
                continue

        self.triplets = torch.LongTensor(self.triplets)

        print(f"    📊 最终有效三元组: {len(self.triplets)}")
        if skipped > 0:
            print(f"    ⚠️ 跳过了 {skipped} 条数据 (正常现象)")

        if len(self.triplets) == 0:
            raise ValueError("❌ 错误：没有生成任何有效数据！请检查 entity2id 是否匹配。")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        return self.triplets[idx]


# ================= 🧠 DistMult 模型 (通用) =================
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embed_dim):
        super(DistMult, self).__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)

        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

        self.criterion = nn.MarginRankingLoss(margin=1.0)

    def forward(self, h, r, t):
        h_e = self.ent_emb(h)
        r_e = self.rel_emb(r)
        t_e = self.ent_emb(t)
        score = torch.sum(h_e * r_e * t_e, dim=1)
        return score

    def calculate_loss(self, h, r, t):
        batch_size = h.size(0)
        neg_t = torch.randint(0, self.num_entities, (batch_size,), device=h.device)
        pos_score = self.forward(h, r, t)
        neg_score = self.forward(h, r, neg_t)
        target = torch.ones(batch_size, device=h.device)
        loss = self.criterion(pos_score, neg_score, target)
        return loss


# ================= 🏃 主训练循环 =================
def train():
    # 1. 检查文件
    if not os.path.exists(ENTITY2ID_FILE):
        print(f"❌ 找不到 {ENTITY2ID_FILE}，请先运行 build_aibl_primekg.py")
        return

    # 2. 加载映射
    print("📥 正在加载 ID 映射表...")
    with open(ENTITY2ID_FILE, 'r') as f:
        entity2id = json.load(f)
    with open(RELATION2ID_FILE, 'r') as f:
        relation2id = json.load(f)

    num_ents = len(entity2id)
    num_rels = len(relation2id)
    print(f"    实体总数: {num_ents} | 关系总数: {num_rels}")

    # 3. 准备数据
    dataset = KGDataset(TRIPLETS_FILE, entity2id, relation2id)

    train_size = int(TRAIN_RATIO * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # 4. 初始化模型
    model = DistMult(num_ents, num_rels, EMBED_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\n🔥 [AIBL] 开始训练 (共 {NUM_EPOCHS} 轮)...")

    # 5. 循环
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=False)

        for batch in progress:
            batch = batch.to(DEVICE)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

            optimizer.zero_grad()
            loss = model.calculate_loss(h, r, t)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        if (epoch) % 1 == 0:
            model.eval()
            test_loss = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(DEVICE)
                    h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
                    loss = model.calculate_loss(h, r, t)
                    test_loss += loss.item()
            avg_test = test_loss / len(test_loader)
            print(f"Epoch {epoch + 1:03d} | 📉 Train Loss: {avg_loss:.4f} | 🔍 Test Loss: {avg_test:.4f}")

    # 6. 保存
    print("\n" + "=" * 40)
    print(f"💾 训练完成！正在保存 AIBL Embeddings 至: {OUTPUT_EMBED}")

    embeddings = model.ent_emb.weight.detach().cpu().numpy()
    np.save(OUTPUT_EMBED, embeddings)

    print(f"✅ 保存成功！矩阵形状: {embeddings.shape}")
    print("=" * 40)


if __name__ == "__main__":
    train()

🚀 [AIBL] 训练设备: cpu
📥 正在加载 ID 映射表...
    实体总数: 7870 | 关系总数: 20
    正在读取图谱文件: aibl_knowledge_triplets.csv ...
    ✅ 成功加载原始数据: 65658 行


Indexing Data: 100%|██████████████████████████████████████████████████████████| 65658/65658 [00:05<00:00, 11173.46it/s]


    📊 最终有效三元组: 65646
    ⚠️ 跳过了 12 条数据 (正常现象)

🔥 [AIBL] 开始训练 (共 100 轮)...


Epoch 001 | 📉 Train Loss: 0.7987 | 🔍 Test Loss: 0.3114


Epoch 002 | 📉 Train Loss: 0.1169 | 🔍 Test Loss: 0.0982


...

Epoch 096 | 📉 Train Loss: 0.0139 | 🔍 Test Loss: 0.0618


Epoch 097 | 📉 Train Loss: 0.0146 | 🔍 Test Loss: 0.0606


Epoch 098 | 📉 Train Loss: 0.0138 | 🔍 Test Loss: 0.0683


Epoch 099 | 📉 Train Loss: 0.0135 | 🔍 Test Loss: 0.0618


Epoch 100 | 📉 Train Loss: 0.0133 | 🔍 Test Loss: 0.0622

💾 训练完成！正在保存 AIBL Embeddings 至: aibl_kg_embeddings.npy
✅ 保存成功！矩阵形状: (7870, 128)
